## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Review scores to letter grades: cut points 60, 70, 80

**Conversion to Categorical Variables** · a worked example from the StatNex teaching guide

**The story.** A company has 37 annual performance-review scores (0 to 100) and wants to turn them into letter grades for the bonus scheme.

**The question.** How many employees land in each grade, and how much do the cut points matter?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `12_utility/categorical_conversion.csv` (37 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Score
75.0
62.0
62.0
80.0
100.0
58.0
34.0
86.0
46.0
85.0
93.0
81.0
71.0
78.0
86.0
84.0
90.0
67.0
60.0
34.0
72.0
82.0
82.0
62.0
74.0
94.0
73.0
83.0
90.0
79.0
83.0
55.0
74.0
74.0
99.0
80.0
79.0
""")

## Libraries

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
# Text and blank cells are left out of the statistics and get no category.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
column = "Score"
original = pd.to_numeric(data[column], errors="coerce")
values = original.dropna()

## Descriptive statistics

In [ ]:
summary = pd.Series({"n": len(values), "mean": values.mean(), "variance": values.var(ddof=1),
                     "SD": values.std(ddof=1), "minimum": values.min(), "maximum": values.max()})
print(summary)

## Boundaries and category names

In [ ]:
# The boundaries and names as typed, separated by commas: k boundaries make k + 1
# categories.
boundaries = [float(text) for text in "60,70,80".replace("、", ",").split(",") if text.strip()]
names = [text.strip() for text in "D,C,B,A".replace("、", ",").split(",") if text.strip()]

# A value sitting exactly on a boundary goes to the class above it ("at least / less
# than"): every class is [lower, upper), closed on the left, which is right=False in
# pandas.cut below. On a grade boundary this decides somebody's grade.
right = False
legend = ([f"under {boundaries[0]:g}"] if boundaries else ["all cases"]) + [
    f"{low:g} to under {high:g}" for low, high in zip(boundaries, boundaries[1:])] + (
    [f"{boundaries[-1]:g} and over"] if boundaries else [])
print(pd.DataFrame({"range": legend, "category": names}).to_string(index=False))

## Converted data

In [ ]:
# pandas.cut sorts each value into its class. The outer edges are -inf and +inf, so a
# value below the first boundary or above the last still gets one; the boundaries
# must rise strictly. ordered=False lets two classes share a name.
classes = pd.cut(values, bins=[-np.inf, *boundaries, np.inf], right=right, labels=names,
                 ordered=False)
converted = classes.astype(object).reindex(data.index)
result = pd.DataFrame({column: original, "category": converted})
result.to_csv("categories.csv", index=False)
print()
print(result.to_string(index=False))

## Frequency distribution of the categories

In [ ]:
counts = converted.value_counts()
frequency = [int(counts.get(name, 0)) for name in names]
distribution = pd.DataFrame({"category": names, "frequency": frequency,
                             "relative frequency": np.array(frequency) / sum(frequency)})
print()
print(distribution.to_string(index=False))

## What the result says

This run cuts the 37 scores at 60, 70 and 80 and names the classes D, C, B and A. The frequency distribution, the count in each class, gives 5 employees a D, 5 a C, 10 a B and 17 an A. The relative frequency of A, its share of all employees, is 0.459, so almost half the staff get an A.

The reason is in the descriptive statistics: the mean score is 74.8, already close to the A line of 80. Also, the A class runs from 80 to the maximum of 100, so very different scores earn the same grade.

Where you put the cut points decides the grades. If the bonus scheme should reward only a few people, the A line must be higher.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.